# Observabilidad de pipeline Kafka + Spark

Notebook P1 para medir disponibilidad, latencia, throughput, logging estructurado y alertas propuestas sobre el pipeline:

`Producer -> Kafka -> Spark Structured Streaming -> evidencias`

Este notebook empieza desde cero y no reemplaza al `07_spark_streaming_consumer_ordenes.ipynb`; lo usa como base conceptual, pero aqui se trabaja la observabilidad de la sesion P1.

## Objetivo

1. Leer eventos desde Kafka.
2. Conservar metadata Kafka: `topic`, `partition`, `offset`, `kafkaTimestamp`.
3. Validar eventos y separar registros validos/invalidos.
4. Calcular `processedAt` y `latencyMs`: `processedAt` es el momento en que Spark procesa el evento; `latencyMs` es la diferencia entre `processedAt` y el `timestamp` enviado por el producer.
5. Estimar throughput con `lastProgress`: `numInputRows`, `inputRowsPerSecond` y `processedRowsPerSecond` describen el ultimo micro-batch procesado.
6. Persistir evidencia observable en Parquet.
7. Relacionar Spark con Prometheus, Grafana y alertas propuestas.

## Datos del entorno

- Kafka broker interno: `ec-kafka:9092`
- Topic: `orden-eventos`
- Prometheus: `http://localhost:49090`
- Grafana: `http://localhost:43000`
- Kafka Exporter: `http://localhost:49308/metrics`
- Salida Parquet: `/opt/artifacts/output/orden_eventos_observabilidad_p1`

## Paso 1. Crear SparkSession

Se carga el conector Kafka para Spark Structured Streaming.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("observabilidad-pipeline-kafka-spark") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-eefe6bb4-cf06-4089-a77e-1c1c5a5bacbe;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#

## Paso 2. Definir parametros y helper de streams

El helper evita dejar queries activas cuando una celda se ejecuta mas de una vez.

In [3]:
KAFKA_BOOTSTRAP_SERVERS = "ec-kafka:9092"
TOPIC_ORDENES = "orden-eventos"
OUTPUT_PATH = "/opt/artifacts/output/orden_eventos_observabilidad_p1"
CHECKPOINT_PATH = "/opt/artifacts/checkpoint/orden-eventos-observabilidad-p1"

def stop_query_if_exists(name):
    query = globals().get(name)
    if query is not None and query.isActive:
        query.stop()
        print(f"{name} detenida.")
    else:
        print(f"{name} no estaba activa.")

def stop_active_streams():
    active = spark.streams.active
    print(f"Queries activas: {len(active)}")
    for query in active:
        print(f"Deteniendo {query.name or query.id}")
        query.stop()

## Paso 3. Leer eventos desde Kafka

`startingOffsets = latest` evita reprocesar backlog antiguo cuando se esta haciendo una prueba en vivo.

In [3]:
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", TOPIC_ORDENES) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

df_kafka.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## Paso 4. Parsear JSON y conservar metadata Kafka

El campo `timestamp` del evento esta en milisegundos Unix y representa cuando el producer genero/publico el evento.

In [4]:
from pyspark.sql.functions import col, from_json, current_timestamp, unix_millis
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

schema_evento = StructType([
    StructField("tipoEvento", StringType(), True),
    StructField("ordenId", LongType(), True),
    StructField("total", DoubleType(), True),
    StructField("estado", StringType(), True),
    StructField("origen", StringType(), True),
    StructField("timestamp", LongType(), True),
])

df_value = df_kafka.select(
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp").alias("kafkaTimestamp"),
    col("value").cast("string").alias("value")
)

df_parsed = df_value.select(
    "topic",
    "partition",
    "offset",
    "kafkaTimestamp",
    from_json(col("value"), schema_evento).alias("evento")
).select("topic", "partition", "offset", "kafkaTimestamp", "evento.*")

df_parsed.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- ordenId: long (nullable = true)
 |-- total: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)



## Paso 5. Crear dataframe observable

- `isValid`: indica si el evento cumple campos minimos.
- `processedAt`: momento de procesamiento en Spark, en ms.
- `latencyMs`: diferencia entre procesamiento y creacion del evento.

In [5]:
df_observable = df_parsed \
    .withColumn(
        "isValid",
        col("tipoEvento").isNotNull()
        & col("ordenId").isNotNull()
        & col("total").isNotNull()
        & col("timestamp").isNotNull()
    ) \
    .withColumn("processedAt", unix_millis(current_timestamp())) \
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))

df_observable.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- ordenId: long (nullable = true)
 |-- total: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = false)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)



## Paso 6. Salida a consola

Arranca un stream visual para generar evidencia rapida. Para que la evidencia Parquet capture los mismos eventos, inicia tambien el Paso 8 antes de producir eventos.

In [6]:
stop_query_if_exists("query_console")

query_console = df_observable.writeStream \
    .queryName("orden_eventos_console_observabilidad") \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start()

# query_console.awaitTermination()

query_console no estaba activa.


26/05/04 09:14:09 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-de969aff-1c50-4aea-8114-2b4f0bfb3710. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/04 09:14:09 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----+---------+------+--------------+----------+-------+-----+------+------+---------+-------+-----------+---------+
|topic|partition|offset|kafkaTimestamp|tipoEvento|ordenId|total|estado|origen|timestamp|isValid|processedAt|latencyMs|
+-----+---------+------+--------------+----------+-------+-----+------+------+---------+-------+-----------+---------+
+-----+---------+------+--------------+----------+-------+-----+------+------+---------+-------+-----------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+---------+------+----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp        |tipoEvento  |ordenId|total|estado   |origen     |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+
|orden-eventos|0        |15    |2026-05-04 09:16:27.53|orden.creada|39     |800.0|PENDIENTE|ec-orden-ms|1777886187513|true   |1777886187834|321      |
+-------------+---------+------+----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+



## Paso 7. Revisar throughput y lag desde Spark

Ejecuta esta celda despues de iniciar Parquet y producir eventos. `lastProgress` muestra el rendimiento del ultimo progreso o ultimo micro-batch.

- `numInputRows`: numero de filas/eventos de entrada en ese micro-batch.
- `inputRowsPerSecond`: filas de entrada por segundo.
- `processedRowsPerSecond`: filas procesadas por segundo.
- `avgOffsetsBehindLatest` y `maxOffsetsBehindLatest`: atraso promedio/maximo respecto al ultimo offset disponible en Kafka.

Si `numInputRows` sale `0`, significa que Spark ejecuto el micro-batch, pero no habia eventos nuevos.

In [7]:
import json

progress = query_console.lastProgress

if progress is None:
    print("Todavia no hay micro-batch procesado. Produce un evento y espera unos segundos.")
else:
    source = progress["sources"][0]
    metrics = source.get("metrics", {})
    status = "idle" if progress["numInputRows"] == 0 else "processed"

    log = {
        "service": "spark-streaming",
        "component": "consumer",
        "topic": TOPIC_ORDENES,
        "batchId": progress["batchId"],
        "numInputRows": progress["numInputRows"],
        "inputRowsPerSecond": progress["inputRowsPerSecond"],
        "processedRowsPerSecond": progress["processedRowsPerSecond"],
        "avgOffsetsBehindLatest": metrics.get("avgOffsetsBehindLatest"),
        "maxOffsetsBehindLatest": metrics.get("maxOffsetsBehindLatest"),
        "status": status
    }
    print(json.dumps(log, indent=2))
    progress

{
  "service": "spark-streaming",
  "component": "consumer",
  "topic": "orden-eventos",
  "batchId": 2,
  "numInputRows": 0,
  "inputRowsPerSecond": 0.0,
  "processedRowsPerSecond": 0.0,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0",
  "status": "idle"
}


### Interpretacion del resultado del Paso 7

Si el resultado muestra `numInputRows: 0`, Spark esta activo y observando Kafka, pero en ese ultimo micro-batch no encontro eventos nuevos.

- `batchId`: numero del micro-batch ejecutado.
- `numInputRows`: eventos recibidos en ese micro-batch.
- `inputRowsPerSecond`: velocidad de entrada de eventos.
- `processedRowsPerSecond`: velocidad de procesamiento de Spark.
- `avgOffsetsBehindLatest` y `maxOffsetsBehindLatest`: atraso respecto al ultimo offset disponible en Kafka.
- `status`: estado calculado para el micro-batch. `idle` significa sin eventos nuevos; `processed` significa que Spark procesó eventos.

Ejemplo: si `numInputRows`, `inputRowsPerSecond` y `processedRowsPerSecond` salen `0`, significa que no llegaron eventos nuevos en ese intervalo. Si ademas `maxOffsetsBehindLatest` es `0`, el consumer esta al dia.

### Evidencia alternativa: Spark UI

Abre `http://localhost:4040` y entra a `Structured Streaming -> Streaming Query Statistics`.

| Seccion Spark UI | Que indica |
|---|---|
| `Input Rate` | velocidad de entrada de eventos hacia Spark |
| `Process Rate` | velocidad con la que Spark procesa eventos |
| `Input Rows` | cantidad de filas/eventos por micro-batch |
| `Batch Duration` | tiempo que tarda cada micro-batch |
| `Operation Duration` | desglose del tiempo usado por operaciones internas |

Lectura rapida: si `Input Rows` es `0`, no llegaron eventos en ese micro-batch. Si `Process Rate` es mayor que `Input Rate`, Spark procesa mas rapido de lo que entran los eventos.

## Paso 8. Persistir evidencia observable en Parquet

Esta salida guarda los eventos procesados con metadata Kafka y campos de observabilidad.

In [8]:
stop_query_if_exists("query_parquet")

query_parquet = df_observable.writeStream \
    .queryName("orden_eventos_parquet_observabilidad_p1") \
    .format("parquet") \
    .option("path", OUTPUT_PATH) \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .outputMode("append") \
    .start()

query_parquet no estaba activa.


26/05/04 09:25:02 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 2
-------------------------------------------
+-------------+---------+------+-----------------------+------------+-------+-----+---------+------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento  |ordenId|total|estado   |origen|timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+------------+-------+-----+---------+------+-------------+-------+-------------+---------+
|orden-eventos|0        |16    |2026-05-04 09:25:58.399|orden.creada|33     |468.0|PENDIENTE|python|1777886758391|true   |1777886758441|50       |
+-------------+---------+------+-----------------------+------------+-------+-----+---------+------+-------------+-------+-------------+---------+



-------------------------------------------
Batch: 3
-------------------------------------------
+-------------+---------+------+-----------------------+------------+-------+-----+---------+------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento  |ordenId|total|estado   |origen|timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+------------+-------+-----+---------+------+-------------+-------+-------------+---------+
|orden-eventos|0        |17    |2026-05-04 09:26:00.451|orden.creada|232    |300.0|PENDIENTE|python|1777886760450|true   |1777886760463|13       |
+-------------+---------+------+-----------------------+------------+-------+-----+---------+------+-------------+-------+-------------+---------+



-------------------------------------------
Batch: 4
-------------------------------------------
+-------------+---------+------+-----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento  |ordenId|total|estado   |origen     |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+
|orden-eventos|0        |18    |2026-05-04 09:26:15.733|orden.creada|40     |900.0|PENDIENTE|ec-orden-ms|1777886775724|true   |1777886776048|324      |
+-------------+---------+------+-----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+



## Paso 8.1. Producir eventos

Con `query_console` y `query_parquet` activos, genera eventos desde `ec-orden-py` o `ec-orden-ms`.

Ejemplo con `ec-orden-ms` desde PowerShell:

```powershell
Invoke-RestMethod `
  -Method Post `
  -Uri "http://localhost:19021/ordenes" `
  -ContentType "application/json" `
  -Body '{"usuarioId":2,"total":600}'
```

Despues espera unos segundos y vuelve a ejecutar el Paso 7 para revisar throughput.

## Paso 9. Leer evidencia guardada

Ejecuta esta celda despues de producir eventos y esperar algunos segundos.

In [10]:
#Leer lo guardado
#df_final = spark.read.parquet("data/orden_eventos_parquet")
#df_final.show(truncate=False)

df_final = spark.read.parquet(OUTPUT_PATH)
df_final.orderBy(col("offset").desc()).show(truncate=False)

+-------------+---------+------+-----------------------+------------+-------+------+---------+-----------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento  |ordenId|total |estado   |origen     |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+------------+-------+------+---------+-----------+-------------+-------+-------------+---------+
|orden-eventos|0        |19    |2026-05-04 09:27:40.658|orden.creada|41     |1600.0|PENDIENTE|ec-orden-ms|1777886860658|true   |1777886860893|235      |
|orden-eventos|0        |18    |2026-05-04 09:26:15.733|orden.creada|40     |900.0 |PENDIENTE|ec-orden-ms|1777886775724|true   |1777886776046|322      |
|orden-eventos|0        |17    |2026-05-04 09:26:00.451|orden.creada|232    |300.0 |PENDIENTE|python     |1777886760450|true   |1777886761236|786      |
|orden-eventos|0        |16    |2026-05-04 09:25:58.399|orden.creada|33     |468.0

## Paso 10. Prometheus

Abre `http://localhost:49090` y revisa:

1. `Status > Targets`: los targets `prometheus` y `kafka-exporter` deben estar `UP`.
2. Ejecuta estas consultas:

```promql
kafka_brokers
up{job="kafka-exporter"}
kafka_broker_info
kafka_consumergroup_lag
```

Si `kafka_consumergroup_lag` no devuelve datos, usa un consumer group activo y vuelve a consultar.

Interpretacion rapida:

- `kafka_broker_info = 1`: Kafka Exporter detecta el broker Kafka.
- `up{job="kafka-exporter"} = 1`: Prometheus pudo consultar el exporter.
- `up{job="kafka-exporter"} = 0`: el exporter esta caido o Prometheus no lo alcanza.
- `kafka_consumergroup_lag = 0`: el consumer group esta al dia.
- `kafka_consumergroup_lag > 0`: hay eventos pendientes por consumir.

## Paso 11. Grafana

Abre `http://localhost:43000` con `admin` / `admin`.

Crea un dashboard minimo:

| Panel | Consulta | Visualizacion |
|---|---|---|
| Kafka Brokers | `kafka_brokers` | Stat |
| Kafka Exporter Up | `up{job="kafka-exporter"}` | Stat |
| Kafka Broker Info | `kafka_broker_info` | Table |
| Consumer Lag | `kafka_consumergroup_lag` | Time series o Table |

### Interpretacion del panel Consumer Lag

Si usas la consulta `kafka_consumergroup_lag`, Grafana dibuja una linea por cada `consumergroup`.

- Valor `0`: el consumer group esta al dia; no tiene mensajes pendientes.
- Valor `1`: hay 1 mensaje pendiente por consumir en ese momento.
- Pico corto de `1` y luego vuelve a `0`: entro un evento, el consumer se atraso momentaneamente y luego lo consumio.
- Valor mayor que `0` sostenido: el consumer no esta alcanzando el ritmo de produccion o esta detenido.

En un panel como `Consumer Lag`, una linea plana en `0` es buena senal. Un pico pequeno y temporal es normal durante la llegada de eventos. Lo preocupante es que el lag crezca o se mantenga alto.

Adjunta capturas de `Explore` y del dashboard.

## Paso 12. Alertas propuestas

| Situacion | Regla propuesta | Accion |
|---|---|---|
| Kafka no visible | `kafka_brokers < 1` | Revisar contenedor Kafka |
| Exporter caido | `up{job="kafka-exporter"} == 0` | Revisar exporter y red Docker |
| Lag alto | `kafka_consumergroup_lag > 100` | Revisar consumer, particiones y carga |
| Eventos invalidos | `invalidRows > 0` | Revisar contrato del evento |

Para esta practica, las alertas pueden entregarse como propuesta documentada. Si deseas configurarlas en Grafana Alerting, usa este flujo minimo:

### Crear una alerta en Grafana

1. Entra a `Alerting -> Alert rules`.
2. Selecciona `New alert rule`.
3. Define el nombre, por ejemplo `Lag alto en Kafka`.
4. En `Query`, selecciona el datasource `Prometheus`.
5. Usa una consulta, por ejemplo:

```promql
kafka_consumergroup_lag > 100
```

6. En `Evaluate every`, usa `1m`.
7. En `For`, usa `2m` para evitar alertas por picos muy breves.
8. En condicion, deja que la regla dispare cuando el resultado sea mayor que `0`.
9. Guarda la regla.

Otra alerta util:

```promql
up{job="kafka-exporter"} == 0
```

Interpretacion: si se cumple, Prometheus no puede consultar el Kafka Exporter.

### Ejemplo 2 opcional: latencia calculada en Spark

`latencyMs` no esta en Prometheus por defecto. En esta practica se calcula en Spark y se revisa desde el notebook o desde Parquet.

| Situacion | Regla operativa | Donde se evalua |
|---|---|---|
| Latencia alta sensible | `latencyMs > 100` | Notebook Spark / Parquet |
| Latencia alta critica | `latencyMs > 1000` | Notebook Spark / Parquet |

Ejemplo:

```python
df_final.filter("latencyMs > 100").show()
```

Resumen: Grafana/Prometheus cubre `up`, `kafka_brokers` y `kafka_consumergroup_lag`; Spark/Parquet cubre `latencyMs`, eventos invalidos y throughput del micro-batch.

In [11]:
df_final.filter("latencyMs > 100").show()


+-------------+---------+------+--------------------+------------+-------+------+---------+-----------+-------------+-------+-------------+---------+
|        topic|partition|offset|      kafkaTimestamp|  tipoEvento|ordenId| total|   estado|     origen|    timestamp|isValid|  processedAt|latencyMs|
+-------------+---------+------+--------------------+------------+-------+------+---------+-----------+-------------+-------+-------------+---------+
|orden-eventos|        0|    18|2026-05-04 09:26:...|orden.creada|     40| 900.0|PENDIENTE|ec-orden-ms|1777886775724|   true|1777886776046|      322|
|orden-eventos|        0|    19|2026-05-04 09:27:...|orden.creada|     41|1600.0|PENDIENTE|ec-orden-ms|1777886860658|   true|1777886860893|      235|
|orden-eventos|        0|    17|2026-05-04 09:26:...|orden.creada|    232| 300.0|PENDIENTE|     python|1777886760450|   true|1777886761236|      786|
+-------------+---------+------+--------------------+------------+-------+------+---------+---------

In [7]:
#Próxima clase: ML

from pyspark.sql.functions import col, window, avg, min as spark_min, max as spark_max, count

df = spark.read.parquet("/opt/artifacts/output/orden_eventos_observabilidad_p1")

serie_latency = df.groupBy(
    window(col("kafkaTimestamp"), "1 minute")
).agg(
    count("*").alias("numEventos"),
    avg("latencyMs").alias("avgLatencyMs"),
    spark_min("latencyMs").alias("minLatencyMs"),
    spark_max("latencyMs").alias("maxLatencyMs")
).orderBy("window")

serie_latency.show(100, truncate=False)


+------------------------------------------+----------+------------+------------+------------+
|window                                    |numEventos|avgLatencyMs|minLatencyMs|maxLatencyMs|
+------------------------------------------+----------+------------+------------+------------+
|{2026-05-04 09:25:00, 2026-05-04 09:26:00}|1         |50.0        |50          |50          |
|{2026-05-04 09:26:00, 2026-05-04 09:27:00}|2         |554.0       |322         |786         |
|{2026-05-04 09:27:00, 2026-05-04 09:28:00}|1         |235.0       |235         |235         |
|{2026-05-04 10:48:00, 2026-05-04 10:49:00}|4         |1165.0      |101         |2790        |
+------------------------------------------+----------+------------+------------+------------+



In [5]:
serie_latency_limpia = serie_latency.select(
    col("window.start").alias("inicioVentana"),
    col("window.end").alias("finVentana"),
    "numEventos",
    "avgLatencyMs",
    "minLatencyMs",
    "maxLatencyMs"
)

serie_latency_limpia.show(100, truncate=False)


+-------------------+-------------------+----------+------------+------------+------------+
|inicioVentana      |finVentana         |numEventos|avgLatencyMs|minLatencyMs|maxLatencyMs|
+-------------------+-------------------+----------+------------+------------+------------+
|2026-05-04 09:25:00|2026-05-04 09:26:00|1         |50.0        |50          |50          |
|2026-05-04 09:26:00|2026-05-04 09:27:00|2         |554.0       |322         |786         |
|2026-05-04 09:27:00|2026-05-04 09:28:00|1         |235.0       |235         |235         |
|2026-05-04 10:48:00|2026-05-04 10:49:00|4         |1165.0      |101         |2790        |
+-------------------+-------------------+----------+------------+------------+------------+



In [ ]:

#avgLatencyMs = latencia real promedio del minuto
#predictedLatencyMs = predicción usando los 3 minutos anteriores
#errorMs = diferencia entre real y predicho

from pyspark.sql.functions import col, window, avg, count
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

df = spark.read.parquet("/opt/artifacts/output/orden_eventos_observabilidad_p1")

serie = df.groupBy(
    window(col("kafkaTimestamp"), "1 minute")
).agg(
    count("*").alias("numEventos"),
    avg("latencyMs").alias("avgLatencyMs")
).select(
    col("window.start").alias("inicioVentana"),
    col("window.end").alias("finVentana"),
    "numEventos",
    "avgLatencyMs"
).orderBy("inicioVentana")

w = Window.orderBy("inicioVentana").rowsBetween(-3, -1)

serie_prediccion = serie.withColumn(
    "predictedLatencyMs",
    avg("avgLatencyMs").over(w)
).withColumn(
    "errorMs",
    col("avgLatencyMs") - col("predictedLatencyMs")
)

serie_prediccion.show(100, truncate=False)


## Paso 13. Detener streams

Al terminar la practica, detiene las queries activas para evitar conflictos con checkpoints o salidas Parquet.

In [ ]:
stop_active_streams()